# Module 5 Lab: One Line or Many?
## 6.3710x — Probability and Statistical Data Analysis

### Where We Left Off

In Module 4, PCA revealed the hidden variable in the crab dataset: **species**. What first looked like one broad cloud was better understood as two populations. You also used AIC to formalize the idea that a richer model can be justified when the gain in fit outweighs the penalty for complexity.

But PCA is an unsupervised tool. It told you about variation, not about **prediction**.

### What's Changed

In this lab, we move from covariance structure to **regression**.

We'll ask a different kind of scientific question:

> Can one crab measurement be predicted from another by a single common linear relationship, or do sex and species change that relationship?

This is the capstone move of the course: you will build a model, test its coefficients, diagnose its failures, and then decide whether a richer model is warranted.

### The Plan

We'll use the crab measurements to study the relationship between frontal lobe size `FL` and carapace length `CL`.

Why this pair?

Earlier in the course, ratios involving `FL` and `CL` hinted that something interesting was happening. Now we'll revisit that same scientific story through the lens of regression.

We'll proceed in stages:

1. fit a **pooled simple regression**
2. quantify uncertainty for the slope
3. inspect residuals and discover what the pooled model hides
4. add **species** and **sex** as predictors
5. test whether the richer model is statistically justified
6. allow **species-specific slopes**
7. compare models from both the **inference** and **prediction** viewpoints

### What's New Conceptually

- **Simple linear regression** as a best linear predictor
- **Multiple regression** with indicator variables
- **Confidence intervals** and **$t$-tests** for regression coefficients
- **Nested-model $F$-tests**
- **Interaction terms** and what they mean scientifically
- **Residual diagnostics**
- **Cross-validation** for comparing predictive performance
- **Pooling pitfalls**: significance is not the same as adequacy, and association is not causation


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import t, f, probplot, mannwhitneyu

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (8, 5)
plt.rcParams["font.size"] = 11

np.random.seed(42)

def load_data(filename):
    """Load CSV from local directory or GitHub."""
    if os.path.exists(filename):
        return pd.read_csv(filename)
    base_url = ("https://raw.githubusercontent.com/"
                "codey-m/prob-stats-labs/main/")
    url = base_url + filename
    print(f"Loading from GitHub: {filename}")
    return pd.read_csv(url)

def add_intercept(X):
    """Add an intercept column of ones to a 1D or 2D array."""
    X = np.asarray(X)
    if X.ndim == 1:
        X = X.reshape(-1, 1)
    return np.column_stack([np.ones(X.shape[0]), X])

def fit_ols(X, y):
    """Fit OLS using matrix formulas and return common summaries."""
    X = np.asarray(X)
    y = np.asarray(y)

    n, p = X.shape
    XtX = X.T @ X
    Xty = X.T @ y

    beta = np.linalg.solve(XtX, Xty)
    y_hat = X @ beta
    residuals = y - y_hat

    RSS = np.sum(residuals**2)
    TSS = np.sum((y - y.mean())**2)
    R2 = 1 - RSS / TSS

    sigma2_hat = RSS / (n - p)
    XtX_inv = np.linalg.inv(XtX)
    cov_beta = sigma2_hat * XtX_inv
    se_beta = np.sqrt(np.diag(cov_beta))

    return {
        "beta": beta,
        "y_hat": y_hat,
        "residuals": residuals,
        "RSS": RSS,
        "TSS": TSS,
        "R2": R2,
        "sigma2_hat": sigma2_hat,
        "cov_beta": cov_beta,
        "se_beta": se_beta,
        "n": n,
        "p": p
    }

def nested_f_test(RSS_reduced, RSS_full, n, p_reduced, p_full):
    """Compare nested linear models via an F-test."""
    q = p_full - p_reduced
    F_stat = ((RSS_reduced - RSS_full) / q) / (RSS_full / (n - p_full))
    p_value = f.sf(F_stat, q, n - p_full)
    return F_stat, p_value

def k_fold_cv_mse(X, y, k=5, seed=42):
    """Compute k-fold CV mean squared error for OLS."""
    rng = np.random.default_rng(seed)
    n = len(y)
    indices = rng.permutation(n)
    folds = np.array_split(indices, k)

    fold_mse = []

    for j in range(k):
        test_idx = folds[j]
        train_idx = np.concatenate([folds[i] for i in range(k) if i != j])

        X_train = X[train_idx]
        y_train = y[train_idx]
        X_test = X[test_idx]
        y_test = y[test_idx]

        model = fit_ols(X_train, y_train)
        y_pred = X_test @ model["beta"]
        mse = np.mean((y_test - y_pred)**2)
        fold_mse.append(mse)

    return np.mean(fold_mse)

def fit_ridge(X, y, lam):
    """Fit ridge regression with an unpenalized intercept."""
    X = np.asarray(X)
    y = np.asarray(y)
    n, p = X.shape

    XtX = X.T @ X
    Xty = X.T @ y

    penalty = np.eye(p)
    penalty[0, 0] = 0.0

    beta = np.linalg.solve(XtX + lam * penalty, Xty)
    return beta

## Part 1: From Clouds to Lines

Let's return to the crab dataset. This time, instead of asking about overall covariance structure, we'll assign one variable the role of **predictor** and another the role of **response**.

We'll use:

- predictor: `CL` (carapace length)
- response: `FL` (frontal lobe size)

This is a natural pair because earlier in the course, the ratio `FL / CL` hinted at hidden structure.


In [ ]:
crabs = load_data("crabs.csv")

print("Shape:", crabs.shape)
print("Columns:", list(crabs.columns))
crabs.head()

In [ ]:
measurement_cols = ["FL", "RW", "CL", "CW", "BD"]
crabs[measurement_cols].describe().round(2)

### TODO 1a

Extract:

- the predictor array $x$ from column `CL`
- the response array $y$ from column `FL`

Also record the sample size $n$.


In [ ]:
# TODO 1a: Extract predictor and response
x = ...
y = ...
n = ...

print(f"n = {n}")
print(f"x shape = {x.shape}")
print(f"y shape = {y.shape}")

### TODO 1b

Build the pooled design matrix for simple regression:

$$
X_\text{pooled} =
\begin{bmatrix}
1 & x_1 \\
1 & x_2 \\
\vdots & \vdots \\
1 & x_n
\end{bmatrix}
$$


In [ ]:
# TODO 1b: Build pooled design matrix
X_pooled = ...

print(f"Shape of X_pooled: {X_pooled.shape}")

In [ ]:
# Provided: first look at the regression problem
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharex=True, sharey=True)

# Left: colored by sex
for sex, color, marker in [("M", "steelblue", "o"),
                           ("F", "coral", "s")]:
    mask = crabs["sex"] == sex
    axes[0].scatter(crabs.loc[mask, "CL"], crabs.loc[mask, "FL"],
                    c=color, marker=marker, s=35,
                    alpha=0.7, edgecolor="white", linewidth=0.3,
                    label=sex)
axes[0].set_title("FL vs CL — Colored by Sex",
                  fontsize=13, fontweight="bold")
axes[0].set_xlabel("CL")
axes[0].set_ylabel("FL")
axes[0].legend(fontsize=10)

# Right: colored by species
for sp, color, label in [("B", "dodgerblue", "Blue"),
                         ("O", "darkorange", "Orange")]:
    mask = crabs["sp"] == sp
    axes[1].scatter(crabs.loc[mask, "CL"], crabs.loc[mask, "FL"],
                    c=color, s=35, alpha=0.7,
                    edgecolor="white", linewidth=0.3,
                    label=label)
axes[1].set_title("FL vs CL — Colored by Species",
                  fontsize=13, fontweight="bold")
axes[1].set_xlabel("CL")
axes[1].set_ylabel("FL")
axes[1].legend(fontsize=10)

plt.tight_layout()
plt.show()

Let's reflect on what you see:

- Is there a strong linear trend in the pooled data?
- Do the sex groups appear separated?
- Do the species groups appear to follow the same line?

At this point, one pooled line might look plausible — but the colored plots already hint that the story may be more subtle.


## Part 2: Pooled Simple Regression

We begin with the simplest model:

$$
Y = \beta_0 + \beta_1 X + Z
$$

This says that all crabs, regardless of species or sex, share one common linear relationship between `CL` and `FL`.


### TODO 2a

Compute the pooled regression coefficients using the simple regression formulas:

$$
\hat\beta_1 =
\frac{\sum_{i=1}^n (x_i - \bar{x})(y_i - \bar{y})}
{\sum_{i=1}^n (x_i - \bar{x})^2}
$$

and

$$
\hat\beta_0 = \bar{y} - \hat\beta_1 \bar{x}
$$


In [ ]:
# TODO 2a: Simple-regression formulas
x_bar = ...
y_bar = ...

beta1_pooled = ...
beta0_pooled = ...

beta_pooled = np.array([beta0_pooled, beta1_pooled])

print(f"Pooled intercept: {beta0_pooled:.4f}")
print(f"Pooled slope:     {beta1_pooled:.4f}")

### TODO 2b

Verify your answer using the matrix OLS formula:

$$
\hat\beta = (X^T X)^{-1} X^T y
$$


In [ ]:
# TODO 2b: Matrix verification
beta_pooled_matrix = ...

print("By scalar formulas: ", np.round(beta_pooled, 6))
print("By matrix formula:  ", np.round(beta_pooled_matrix, 6))

In [ ]:
# Checkpoint
assert np.allclose(beta_pooled, beta_pooled_matrix), \
    "Your scalar and matrix OLS estimates do not match."
print("✓ Scalar and matrix regression estimates agree.")

### TODO 2c

Compute:

- fitted values
- residuals
- residual sum of squares (RSS)
- total sum of squares (TSS)
- $R^2$

Recall:

$$
R^2 = 1 - \frac{\mathrm{RSS}}{\mathrm{TSS}}
$$


In [ ]:
# TODO 2c: Fit summaries for pooled model
y_hat_pooled = ...
residuals_pooled = ...

RSS_pooled = ...
TSS_pooled = ...
R2_pooled = ...

print(f"RSS (pooled): {RSS_pooled:.4f}")
print(f"TSS:          {TSS_pooled:.4f}")
print(f"R^2 (pooled): {R2_pooled:.4f}")

In [ ]:
# Provided: get the full pooled OLS summary via helper
model_pooled = fit_ols(X_pooled, y)

print("Pooled coefficients from helper:")
print(np.round(model_pooled["beta"], 6))
print(f"R^2 from helper: {model_pooled['R2']:.4f}")

### TODO 2d

Compute the standard error, 95% confidence interval, and $t$-statistic for the pooled slope.

For simple regression with intercept, the residual degrees of freedom are $n - 2$.


In [ ]:
# TODO 2d: Inference for the pooled slope
se_beta1_pooled = ...
df_pooled = ...

t_stat_beta1_pooled = ...
p_value_beta1_pooled = ...

t_crit_pooled = ...
ci_beta1_pooled_lower = ...
ci_beta1_pooled_upper = ...

print(f"SE(slope):     {se_beta1_pooled:.5f}")
print(f"t-statistic:   {t_stat_beta1_pooled:.3f}")
print(f"p-value:       {p_value_beta1_pooled:.3e}")
print(f"95% CI slope:  [{ci_beta1_pooled_lower:.4f}, "
      f"{ci_beta1_pooled_upper:.4f}]")

In [ ]:
# Provided: pooled regression line
x_grid = np.linspace(x.min(), x.max(), 200)
y_grid_pooled = beta0_pooled + beta1_pooled * x_grid

fig, ax = plt.subplots(figsize=(8, 6))

for sp, color, label in [("B", "dodgerblue", "Blue"),
                         ("O", "darkorange", "Orange")]:
    mask = crabs["sp"] == sp
    ax.scatter(crabs.loc[mask, "CL"], crabs.loc[mask, "FL"],
               c=color, s=35, alpha=0.65,
               edgecolor="white", linewidth=0.3,
               label=label)

ax.plot(x_grid, y_grid_pooled, color="black", linewidth=2.5,
        label="Pooled regression line")

ax.set_xlabel("CL")
ax.set_ylabel("FL")
ax.set_title("Pooled Simple Regression",
             fontsize=13, fontweight="bold")
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

**What to notice:**

The pooled slope will likely be highly significant. But a small p-value for the slope only says there is evidence of **some linear association** between `CL` and `FL`.

It does **not** say that one common line is an adequate model for all crabs.

That distinction is the heart of the rest of this lab.

---


## Part 3: Diagnostics — What Pooling Hides

Residuals tell you what the model failed to explain.

If one common line really were adequate, the residuals should look like noise — centered around zero, with no systematic structure.

Let's check.


In [ ]:
# Provided: residual plot colored by species
fig, ax = plt.subplots(figsize=(8, 5))

for sp, color, label in [("B", "dodgerblue", "Blue"),
                         ("O", "darkorange", "Orange")]:
    mask = crabs["sp"] == sp
    ax.scatter(crabs.loc[mask, "CL"], residuals_pooled[mask],
               c=color, s=35, alpha=0.7,
               edgecolor="white", linewidth=0.3,
               label=label)

ax.axhline(0, color="black", linewidth=1.5, linestyle="--")
ax.set_xlabel("CL")
ax.set_ylabel("Residual")
ax.set_title("Residuals from the Pooled Model — Colored by Species",
             fontsize=13, fontweight="bold")
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

In [ ]:
# Provided: QQ plot of pooled residuals
fig, ax = plt.subplots(figsize=(6, 5))
probplot(residuals_pooled, dist="norm", plot=ax)
ax.set_title("QQ Plot of Pooled Residuals",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

### TODO 3a

Compute the mean pooled residual within each species and within each sex.

If the pooled model misses systematic group structure, these groupwise means should not all be close to zero.


In [ ]:
# TODO 3a: Residual means by group
resid_df = crabs[["sp", "sex"]].copy()
resid_df["resid_pooled"] = residuals_pooled

mean_resid_by_species = ...
mean_resid_by_sex = ...

print("Mean residual by species:")
print(mean_resid_by_species.round(4))
print()
print("Mean residual by sex:")
print(mean_resid_by_sex.round(4))

In [ ]:
# Provided: boxplots of residuals by group
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)

sns.boxplot(data=resid_df, x="sp", y="resid_pooled",
            palette=["dodgerblue", "darkorange"], ax=axes[0])
axes[0].axhline(0, color="black", linestyle="--", linewidth=1)
axes[0].set_title("Residuals by Species",
                  fontsize=12, fontweight="bold")
axes[0].set_xlabel("Species")
axes[0].set_ylabel("Pooled residual")

sns.boxplot(data=resid_df, x="sex", y="resid_pooled",
            palette=["coral", "steelblue"], ax=axes[1])
axes[1].axhline(0, color="black", linestyle="--", linewidth=1)
axes[1].set_title("Residuals by Sex",
                  fontsize=12, fontweight="bold")
axes[1].set_xlabel("Sex")
axes[1].set_ylabel("Pooled residual")

plt.tight_layout()
plt.show()

Let's reflect on this:

1. Can the pooled slope be highly significant even if the pooled model is inadequate?
2. Which grouping variable seems more strongly associated with systematic residual shifts?
3. Why is this a warning against interpreting a single regression line too quickly?

The next step is to let the model account for group structure directly.

---


## Part 4: Multiple Regression with Group Indicators

The pooled model forced all crabs onto one common line.

Now we'll allow the line to shift depending on species and sex.

We begin with the additive model:

$$
Y = \beta_0 + \beta_1 X + \beta_2 I_{\text{Blue}}
+ \beta_3 I_{\text{Female}} + Z
$$

Here:

- $I_{\text{Blue}} = 1$ for Blue crabs, 0 otherwise
- $I_{\text{Female}} = 1$ for Female crabs, 0 otherwise

This model allows different **intercepts** by group, but it still assumes one common slope for `CL`.


### TODO 4a

Create the indicator variables:

- `is_blue`
- `is_female`

Then build the additive design matrix:

$$
X_\text{add} =
\begin{bmatrix}
1 & x_i & I_{\text{Blue},i} & I_{\text{Female},i}
\end{bmatrix}_{i=1}^n
$$


In [ ]:
# TODO 4a: Group indicators and additive design matrix
is_blue = ...
is_female = ...

X_add = ...

print(f"Shape of X_add: {X_add.shape}")

### TODO 4b

Fit the additive model using the OLS helper and extract:

- coefficient estimates
- standard errors
- $R^2$


In [ ]:
# TODO 4b: Fit additive model
model_add = ...
beta_add = ...
se_add = ...
R2_add = ...

print("Additive-model coefficients:")
print(np.round(beta_add, 4))
print(f"R^2 (additive): {R2_add:.4f}")

### TODO 4c

Construct 95% confidence intervals and two-sided p-values for all additive-model coefficients.


In [ ]:
# TODO 4c: Coefficient inference for additive model
df_add = ...
t_crit_add = ...

t_stats_add = ...
p_values_add = ...

ci_add_lower = ...
ci_add_upper = ...

coef_table_add = pd.DataFrame({
    "term": ["Intercept", "CL", "Blue", "Female"],
    "estimate": beta_add,
    "se": se_add,
    "t_stat": t_stats_add,
    "p_value": p_values_add,
    "ci_lower": ci_add_lower,
    "ci_upper": ci_add_upper
})

coef_table_add.round(4)

### TODO 4d

Use a nested-model $F$-test to compare:

- reduced model: pooled simple regression
- full model: additive multiple regression

This tests the joint null hypothesis:

$$
H_0: \beta_2 = \beta_3 = 0
$$

In words: after controlling for `CL`, do species and sex add any explanatory power?


In [ ]:
# TODO 4d: Joint F-test for group effects
F_groups = ...
p_value_groups = ...

print(f"F statistic (pooled vs additive): {F_groups:.4f}")
print(f"p-value:                          {p_value_groups:.3e}")

In [ ]:
# Provided: additive model fit visualized by species and sex
x_grid = np.linspace(x.min(), x.max(), 200)

fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)

for j, sex_label in enumerate(["M", "F"]):
    ax = axes[j]
    sex_val = 1 if sex_label == "F" else 0

    for sp, color, label in [("O", "darkorange", "Orange"),
                             ("B", "dodgerblue", "Blue")]:
        sp_val = 1 if sp == "B" else 0
        mask = (crabs["sex"] == sex_label) & (crabs["sp"] == sp)

        ax.scatter(crabs.loc[mask, "CL"], crabs.loc[mask, "FL"],
                   c=color, s=35, alpha=0.65,
                   edgecolor="white", linewidth=0.3)

        X_grid = np.column_stack([
            np.ones_like(x_grid),
            x_grid,
            np.full_like(x_grid, sp_val),
            np.full_like(x_grid, sex_val)
        ])
        y_grid = X_grid @ beta_add
        ax.plot(x_grid, y_grid, color=color, linewidth=2.5,
                label=f"{label} fit")

    ax.set_title(f"Additive Model — Sex = {sex_label}",
                 fontsize=12, fontweight="bold")
    ax.set_xlabel("CL")
    ax.set_ylabel("FL")
    ax.legend(fontsize=9)

plt.tight_layout()
plt.show()

**What to notice:**

The additive model can shift the line up or down by group, but it still forces all groups to share the same slope.

That gives a more flexible scientific story than the pooled model:

- size matters
- species may shift the relationship
- sex may shift the relationship

But do different species also follow **different slopes**?

---


## Part 5: One Slope or Many?

The additive model allows different intercepts but keeps one common slope.

Now we'll ask whether species changes the slope itself.

We add one interaction term:

$$
Y = \beta_0 + \beta_1 X + \beta_2 I_{\text{Blue}}
+ \beta_3 I_{\text{Female}} + \beta_4 (X \cdot I_{\text{Blue}}) + Z
$$

Interpretation:

- for Orange crabs, the slope is $\beta_1$
- for Blue crabs, the slope is $\beta_1 + \beta_4$

If $\beta_4 = 0$, both species share the same slope.


### TODO 5a

Build the interaction term `x_blue = x * is_blue` and then construct the interaction design matrix.


In [ ]:
# TODO 5a: Interaction design matrix
x_blue = ...
X_inter = ...

print(f"Shape of X_inter: {X_inter.shape}")

### TODO 5b

Fit the interaction model and compute:

- coefficient estimates
- standard errors
- $R^2$
- the implied Orange and Blue slopes


In [ ]:
# TODO 5b: Fit interaction model
model_inter = ...
beta_inter = ...
se_inter = ...
R2_inter = ...

slope_orange = ...
slope_blue = ...

print("Interaction-model coefficients:")
print(np.round(beta_inter, 4))
print()
print(f"R^2 (interaction): {R2_inter:.4f}")
print(f"Orange slope:      {slope_orange:.4f}")
print(f"Blue slope:        {slope_blue:.4f}")

### TODO 5c

Use a nested-model $F$-test to compare:

- reduced model: additive model
- full model: interaction model

This tests:

$$
H_0: \beta_4 = 0
$$

In words: once we allow species and sex intercept shifts, do we still need a species-specific slope?


In [ ]:
# TODO 5c: F-test for the interaction term
F_interaction = ...
p_value_interaction = ...

print(f"F statistic (additive vs interaction): "
      f"{F_interaction:.4f}")
print(f"p-value:                                "
      f"{p_value_interaction:.3e}")

In [ ]:
# Provided: interaction-model fit visualized by sex
x_grid = np.linspace(x.min(), x.max(), 200)

fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)

for j, sex_label in enumerate(["M", "F"]):
    ax = axes[j]
    sex_val = 1 if sex_label == "F" else 0

    for sp, color, label in [("O", "darkorange", "Orange"),
                             ("B", "dodgerblue", "Blue")]:
        sp_val = 1 if sp == "B" else 0
        mask = (crabs["sex"] == sex_label) & (crabs["sp"] == sp)

        ax.scatter(crabs.loc[mask, "CL"], crabs.loc[mask, "FL"],
                   c=color, s=35, alpha=0.65,
                   edgecolor="white", linewidth=0.3)

        X_grid = np.column_stack([
            np.ones_like(x_grid),
            x_grid,
            np.full_like(x_grid, sp_val),
            np.full_like(x_grid, sex_val),
            x_grid * sp_val
        ])
        y_grid = X_grid @ beta_inter
        ax.plot(x_grid, y_grid, color=color, linewidth=2.5,
                label=f"{label} fit")

    ax.set_title(f"Interaction Model — Sex = {sex_label}",
                 fontsize=12, fontweight="bold")
    ax.set_xlabel("CL")
    ax.set_ylabel("FL")
    ax.legend(fontsize=9)

plt.tight_layout()
plt.show()

Let's reflect on this:

1. Does allowing a species-specific slope materially change the fit?
2. Which model seems scientifically more plausible now:
   - one common line,
   - parallel lines,
   - or species-specific slopes?
3. How do the additive and interaction models differ in what they say biologically?

---


## Part 6: Prediction vs Inference

So far, we have mostly been in the **inference** mindset:

- which coefficients matter?
- which model is statistically justified?
- what do the parameters mean?

Now we'll briefly shift into the **prediction** mindset.

A more flexible model nearly always improves training fit. But does it improve out-of-sample prediction?

We'll use 5-fold cross-validation to compare:

1. pooled model
2. additive model
3. interaction model


### TODO 6a

Compute the 5-fold cross-validated mean squared error (CV MSE) for all three models.


In [ ]:
# TODO 6a: Cross-validated MSE
cv_mse_pooled = ...
cv_mse_add = ...
cv_mse_inter = ...

print(f"CV MSE (pooled):      {cv_mse_pooled:.4f}")
print(f"CV MSE (additive):    {cv_mse_add:.4f}")
print(f"CV MSE (interaction): {cv_mse_inter:.4f}")

### TODO 6b

Create a model-comparison table showing:

- number of parameters
- training $R^2$
- CV MSE
- a short qualitative interpretation


In [ ]:
# TODO 6b: Comparison table
comparison_table = pd.DataFrame({
    "model": ["Pooled", "Additive", "Interaction"],
    "parameters": [X_pooled.shape[1], X_add.shape[1], X_inter.shape[1]],
    "train_R2": [R2_pooled, R2_add, R2_inter],
    "cv_mse": [cv_mse_pooled, cv_mse_add, cv_mse_inter],
    "comment": ["...", "...", "..."]
})

comparison_table

**What to notice:**

Inference and prediction are related, but not identical.

A model can be scientifically informative and statistically significant even if its predictive advantage is modest. Conversely, a flexible model can improve training fit without meaningfully improving generalization.

This distinction will matter later in the course project, where you'll build a one-dimensional score and then test it.

---


## Part 7 (Optional): From Regression to a One-Dimensional Score

This short extension previews the logic of the final project.

Instead of predicting one continuous measurement from another, we now try to build a **score** that separates the two species.

We'll use all five measurements as predictors and fit a ridge regression score for the binary target:

$$
G =
\begin{cases}
1 & \text{if Blue} \\
0 & \text{if Orange}
\end{cases}
$$

The fitted score is

$$
\hat s(x) = \hat\beta_0 + \hat\beta^T x
$$

Large values of $\hat s(x)$ should correspond to crabs that look more Blue-like.

This is not yet the final project — but it is the same pattern:

1. learn a score
2. inspect the score distribution
3. test whether one group tends to have larger scores than the other


### TODO 7a

Build a standardized feature matrix using all five measurements, create the binary label `g_blue`, and fit a ridge score with regularization parameter `lam = 1.0`.


In [ ]:
# TODO 7a: Ridge score for species
X_score_raw = crabs[measurement_cols].values
X_score_mean = ...
X_score_std = ...
X_score_stdized = ...

g_blue = ...

X_score = add_intercept(X_score_stdized)
lam = 1.0

beta_ridge = ...
score = ...

print("Ridge score coefficients:")
print(np.round(beta_ridge, 4))

In [ ]:
# Provided: score histograms by species
fig, ax = plt.subplots(figsize=(8, 5))

score_blue = score[g_blue == 1]
score_orange = score[g_blue == 0]

ax.hist(score_orange, bins=25, alpha=0.55, color="darkorange",
        edgecolor="white", label="Orange")
ax.hist(score_blue, bins=25, alpha=0.55, color="dodgerblue",
        edgecolor="white", label="Blue")

ax.set_xlabel("Ridge score")
ax.set_ylabel("Count")
ax.set_title("A One-Dimensional Score for Species",
             fontsize=13, fontweight="bold")
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

### TODO 7b

Use a Wilcoxon rank-sum / Mann-Whitney test to compare the score distributions for Blue and Orange crabs.

Use the alternative that Blue crabs tend to have larger scores.


In [ ]:
# TODO 7b: Mann-Whitney / Wilcoxon on the score
mw_stat, mw_pvalue = ...

print(f"Mann-Whitney statistic: {mw_stat:.2f}")
print(f"p-value:                {mw_pvalue:.3e}")

**Why this matters**

This optional section compresses a multivariate problem into a single learned score and then applies a hypothesis test to the score distribution.

That is the exact high-level pattern you'll use again later in the course project.


## Report Your Results

Run the cell below and enter the values on the course platform. Round as indicated.


In [ ]:
print("=" * 60)
print("MODULE 5 REPORT VALUES")
print("=" * 60)
print(f"R1.  Pooled intercept (4 dec):                 "
      f"{beta0_pooled:.4f}")
print(f"R2.  Pooled slope (4 dec):                     "
      f"{beta1_pooled:.4f}")
print(f"R3.  Pooled R^2 (4 dec):                       "
      f"{R2_pooled:.4f}")
print(f"R4.  Pooled slope t-stat (3 dec):              "
      f"{t_stat_beta1_pooled:.3f}")
print(f"R5.  Pooled slope p-value (3 sig fig):         "
      f"{p_value_beta1_pooled:.3e}")
print(f"R6.  95% CI pooled slope, lower (4 dec):       "
      f"{ci_beta1_pooled_lower:.4f}")
print(f"R7.  95% CI pooled slope, upper (4 dec):       "
      f"{ci_beta1_pooled_upper:.4f}")
print(f"R8.  Mean pooled residual, Blue (4 dec):       "
      f"{mean_resid_by_species['B']:.4f}")
print(f"R9.  Mean pooled residual, Orange (4 dec):     "
      f"{mean_resid_by_species['O']:.4f}")
print(f"R10. Additive coefficient, Blue (4 dec):       "
      f"{beta_add[2]:.4f}")
print(f"R11. Additive coefficient, Female (4 dec):     "
      f"{beta_add[3]:.4f}")
print(f"R12. Additive R^2 (4 dec):                     "
      f"{R2_add:.4f}")
print(f"R13. F statistic pooled vs additive (3 dec):   "
      f"{F_groups:.3f}")
print(f"R14. p-value pooled vs additive (3 sig fig):   "
      f"{p_value_groups:.3e}")
print(f"R15. Interaction coefficient CL×Blue (4 dec):  "
      f"{beta_inter[4]:.4f}")
print(f"R16. Blue slope in interaction model (4 dec):  "
      f"{slope_blue:.4f}")
print(f"R17. Orange slope in interaction model (4 dec):"
      f"{slope_orange:.4f}")
print(f"R18. Interaction R^2 (4 dec):                  "
      f"{R2_inter:.4f}")
print(f"R19. F statistic add vs inter (3 dec):         "
      f"{F_interaction:.3f}")
print(f"R20. p-value add vs inter (3 sig fig):         "
      f"{p_value_interaction:.3e}")
print(f"R21. 5-fold CV MSE pooled (4 dec):             "
      f"{cv_mse_pooled:.4f}")
print(f"R22. 5-fold CV MSE additive (4 dec):           "
      f"{cv_mse_add:.4f}")
print(f"R23. 5-fold CV MSE interaction (4 dec):        "
      f"{cv_mse_inter:.4f}")

## Reflection Questions

Discuss these with the course chatbot or on the discussion forum.


**Q1.** The pooled slope is likely highly significant. Why does that not settle the scientific question of whether one common regression line is adequate?


**Q2.** In the additive model, what does the coefficient on `Blue` mean? What is being held fixed when you interpret it?


**Q3.** What is the scientific difference between an additive group effect and an interaction effect? In words: what does it mean for species to shift the line up or down, versus changing its slope?


**Q4.** Suppose the additive model is strongly favored by the $F$-test, but the cross-validated MSE improves only slightly. What does that tell you about inference versus prediction?


**Q5.** How does this lab connect to the course's broader theme that pooling can hide important structure? Where in the lab did you first see evidence of that?


**Q6.** The optional score section turned a multivariate problem into a one-dimensional score and then tested that score. Why is that strategy useful in more complicated data-analysis problems?


**Q7.** Regression coefficients describe associations. Under what circumstances, if any, would it be reasonable to make a causal statement from a coefficient in a model like this one?
